In [1]:
%pip install qiskit[visualization] hdh


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os, glob
from collections import defaultdict
from qiskit import QuantumCircuit
from hdh.converters import from_qiskit

N            = 1
CIRCUITS_DIR = "./Quantum_Circuits"

## Circuit

In [3]:
all_files = sorted(glob.glob(os.path.join(CIRCUITS_DIR, "monolithic__*.qasm")))
all_files = [f for f in all_files if "__bloated__" not in f]
all_files.sort(key=lambda p: int(os.path.basename(p).replace(".qasm","").split("__")[3]))

circuits = []
for path in all_files[:N]:
    parts = os.path.basename(path).replace(".qasm","").split("__")
    qc    = QuantumCircuit.from_qasm_file(path)
    circuits.append({"identifier": parts[2], "qc": qc})
    print(f"{parts[2]}  —  {qc.num_qubits} qubits, {len(qc.data)} instructions")
    print(qc.draw("text", fold=-1))

ae_indep_tket  —  2 qubits, 12 instructions
            ┌───┐     ┌───────┐                                  ┌───┐        ░ ┌─┐   
  eval: ────┤ H ├─────┤ U1(0) ├──■──────────────────────■────────┤ H ├────────░─┤M├───
        ┌───┴───┴────┐├───────┤┌─┴─┐┌────────────────┐┌─┴─┐┌─────┴───┴──────┐ ░ └╥┘┌─┐
     q: ┤ Ry(0.9273) ├┤ U1(0) ├┤ X ├┤ U3(11.639,0,0) ├┤ X ├┤ U3(0.9273,0,0) ├─░──╫─┤M├
        └────────────┘└───────┘└───┘└────────────────┘└───┘└────────────────┘ ░  ║ └╥┘
meas: 2/═════════════════════════════════════════════════════════════════════════╩══╩═
                                                                                 0  1 


## 1. Telegate
Qubits → vertices. Multi-qubit gates → hyperedges.

In [4]:
def circuit_to_telegate(qc):
    idx  = {q: i for i, q in enumerate(qc.qubits)}
    SKIP = {"barrier", "snapshot", "delay", "label", "measure"}
    edges, names = [], []
    for inst in qc.data:
        if inst.operation.name in SKIP or len(inst.qubits) < 2:
            continue
        e = list(dict.fromkeys(idx[q] for q in inst.qubits))
        if len(e) >= 2:
            edges.append(e)
            names.append(inst.operation.name)
    return list(range(qc.num_qubits)), edges, names

for c in circuits:
    vertices, edges, names = circuit_to_telegate(c["qc"])
    print(f"Vertices (qubits) : {vertices}")
    print(f"Hyperedges        : {len(edges)}")
    for i, (e, n) in enumerate(zip(edges, names)):
        print(f"  e{i:02d}  {n:12s}  qubits={e}")

Vertices (qubits) : [0, 1]
Hyperedges        : 2
  e00  cx            qubits=[0, 1]
  e01  cx            qubits=[0, 1]


## 2. Teledata
Multi-qubit gates → vertices. Qubits → hyperedges (each qubit connects the gates that act on it).

In [5]:
def circuit_to_teledata(qc):
    idx  = {q: i for i, q in enumerate(qc.qubits)}
    SKIP = {"barrier", "snapshot", "delay", "label", "measure"}
    gates = []
    for inst in qc.data:
        if inst.operation.name in SKIP or len(inst.qubits) < 2:
            continue
        qs = list(dict.fromkeys(idx[q] for q in inst.qubits))
        if len(qs) >= 2:
            gates.append((inst.operation.name, qs))
    qubit_to_gates = defaultdict(list)
    for gi, (_, qs) in enumerate(gates):
        for q in qs:
            qubit_to_gates[q].append(gi)
    qubit_ids = sorted(qubit_to_gates)
    return list(range(len(gates))), [gates[i][0] for i in range(len(gates))], qubit_ids, [qubit_to_gates[q] for q in qubit_ids]

for c in circuits:
    vertices, names, qubit_ids, edges = circuit_to_teledata(c["qc"])
    print(f"Vertices (gates)  : {[f'g{i}:{n}' for i, n in enumerate(names)]}")
    print(f"Hyperedges        : {len(edges)}")
    for q, e in zip(qubit_ids, edges):
        print(f"  q{q}  gates={e}")

Vertices (gates)  : ['g0:cx', 'g1:cx']
Hyperedges        : 2
  q0  gates=[0, 1]
  q1  gates=[0, 1]


## 3. HDH
Qubit-state moments → vertices (`q_t`). Gates → typed hyperedges. Metadata tracks quantum vs classical.

In [6]:
for c in circuits:
    h = from_qiskit(c["qc"])
    q_nodes = [n for n in h.S if h.sigma[n] == 'q']
    c_nodes = [n for n in h.S if h.sigma[n] == 'c']
    q_edges = [e for e in h.C if h.tau[e]   == 'q']
    c_edges = [e for e in h.C if h.tau[e]   == 'c']
    print(f"Nodes  : {len(h.S)}  ({len(q_nodes)} quantum, {len(c_nodes)} classical)")
    print(f"Edges  : {len(h.C)}  ({len(q_edges)} quantum, {len(c_edges)} classical)")
    print()
    print("Nodes (id, type, timestep):")
    for n in sorted(h.S):
        print(f"  {n:12s}  {'quantum' if h.sigma[n]=='q' else 'classical':9s}  t={h.time_map[n]}")
    print()
    print("Edges (gate, type, nodes):")
    for e in h.C:
        print(f"  {h.gate_name[e]:12s}  {'quantum' if h.tau[e]=='q' else 'classical':9s}  {sorted(e)}")

Nodes  : 23  (21 quantum, 2 classical)
Edges  : 19  (17 quantum, 2 classical)

Nodes (id, type, timestep):
  c0_t11        classical  t=11
  c1_t11        classical  t=11
  q0_t0         quantum    t=0
  q0_t1         quantum    t=1
  q0_t10        quantum    t=10
  q0_t2         quantum    t=2
  q0_t3         quantum    t=3
  q0_t4         quantum    t=4
  q0_t5         quantum    t=5
  q0_t7         quantum    t=7
  q0_t8         quantum    t=8
  q0_t9         quantum    t=9
  q1_t0         quantum    t=0
  q1_t1         quantum    t=1
  q1_t10        quantum    t=10
  q1_t2         quantum    t=2
  q1_t3         quantum    t=3
  q1_t4         quantum    t=4
  q1_t5         quantum    t=5
  q1_t6         quantum    t=6
  q1_t7         quantum    t=7
  q1_t8         quantum    t=8
  q1_t9         quantum    t=9

Edges (gate, type, nodes):
  h             quantum    ['q0_t0', 'q0_t1']
  cx_stage3     quantum    ['q0_t4', 'q0_t5']
  u3            quantum    ['q1_t5', 'q1_t6']
  cx_stage